# Categorical Bias (CB) Analysis — PersianLLaMA

Computes Categorical Bias (CB) scores for PersianLLaMA using Persian-translated StereoSet sentences.

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)

In [ ]:
MODEL_NAME = 'Narabzad/PersianLLaMA-13B'
tokenizer_fa = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fa = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model_fa.eval()

In [ ]:
def compute_pll(sentence, tokenizer, model, device='cuda'):
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 input_ids = inputs['input_ids']
 with torch.no_grad():
 outputs = model(**inputs, labels=input_ids)
 return -outputs.loss.item() * (input_ids.shape[1] - 1)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pll_fa = []
for _, row in tqdm(df.iterrows(), total=len(df)):
 pll_fa.append(compute_pll(str(row['Persian']), tokenizer_fa, model_fa, device))
df['PLLPersian'] = pll_fa

In [ ]:
cb_rows = []
for context, group in df.groupby('context'):
 s = group[group['label_name']=='stereotype']['PLLPersian'].values[0]
 a = group[group['label_name']=='anti-stereotype']['PLLPersian'].values[0]
 u = group[group['label_name']=='unrelated']['PLLPersian'].values[0]
 denom = abs(s) + abs(a) + abs(u)
 cb = (abs(s) / denom) * 100
 for idx in group.index:
 cb_rows.append((idx, cb))
cb_df = pd.DataFrame(cb_rows, columns=['idx','CBPersian']).set_index('idx')
df['CBPersian'] = cb_df['CBPersian']

In [ ]:
print('CB by bias type (PersianLLaMA):')
print(df.groupby('bias_type')['CBPersian'].mean().round(3))

In [ ]:
df.to_csv('/content/drive/MyDrive/stereoset_persianLLama_ByBiasType.csv')
print('Saved')